# micro-sam GPU segmentation in napari

Interactive + automatic segmentation with [micro-sam](https://github.com/computational-cell-analytics/micro-sam) using napari, running on the GPU.

**Kernel:** run this notebook on **`Python (micro_sam)`** (top-right kernel picker). That env has `micro_sam`, `napari`, and a CUDA build of PyTorch.

**Models** (`model_type`): use the microscopy-finetuned ones for best results —
`vit_b_lm` / `vit_l_lm` (light / fluorescence microscopy), `vit_b_em_organelles` (EM), or the vanilla SAM `vit_b` / `vit_l` / `vit_h`. The GPU is used automatically when available.

In [1]:
# Setup + GPU check
# --- Fix Qt platform plugin ("Could not find the Qt platform plugin windows") ---
# The notebook kernel doesn't run conda's Qt activation script, so QT_QPA_PLATFORM_PLUGIN_PATH
# is empty and Qt can't find qwindows.dll. Point it at the env's Qt6 plugins before importing napari.
import os
import sys

_qt_plugins = os.path.join(sys.prefix, "Library", "lib", "qt6", "plugins")
if os.path.isdir(_qt_plugins):
    os.environ["QT_PLUGIN_PATH"] = _qt_plugins
    os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = os.path.join(_qt_plugins, "platforms")
else:
    print("WARNING: Qt6 plugin dir not found at", _qt_plugins)

import torch
import napari
import micro_sam

print("micro_sam:", micro_sam.__version__)
print("napari   :", napari.__version__)
print("torch    :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device :", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")


micro_sam: 1.8.4
napari   : 0.7.1
torch    : 2.12.0
CUDA available: True
Using device : NVIDIA GeForce RTX 3080


In [8]:
viewer = napari.Viewer()

In [10]:
# Step 2 — fine-tune the Light Microscopy model (vit_b_lm) on your seed labels.
# Phase contrast is light microscopy, so we start from vit_b_lm and train WITH the
# instance-segmentation decoder (needed for good automatic AIS/APG segmentation).
from pathlib import Path
import numpy as np
import imageio.v3 as imageio
from torch.utils.data import random_split
import torch_em
from micro_sam.training import train_sam_for_configuration, default_sam_dataset

# --- your data ---
labels_folder = Path(r"C:\Users\isobe\Downloads\NXD8_png-20260712T141854Z-2-001\NXD8_outputs")
initial_image_folder = Path(r"C:\Users\isobe\Downloads\NXD8_png-20260712T141854Z-2-001\NXD8_png\subset\initial")

# Patch size for training. Must be <= your image size; drop to (256, 256) if images are small.
patch_shape = (512, 512)

# where checkpoints + logs are written: <save_root>/checkpoints/<name>/best.pt
save_root = Path.cwd() / "finetune"
name = "nxd8_vit_b_lm"

# Match each image to its label by sorted filename order.
image_paths = sorted(initial_image_folder.glob("*.png"))
label_paths = sorted(labels_folder.glob("*.tif"))
print(f"images: {len(image_paths)}  |  labels: {len(label_paths)}")
assert len(image_paths) > 0, "No images found - check the folder / pattern."
assert len(image_paths) == len(label_paths), "image/label counts differ - names must correspond."

# Your PNGs are 4-channel RGBA, which micro-sam can't ingest (it needs 1 or 3 channels)
# and which also made the loader stack them into one volume. Load each image, drop the
# alpha channel, and collapse RGB -> single-channel grayscale (phase contrast is grayscale).
def load_gray(p):
    im = imageio.imread(p)
    if im.ndim == 3:              # (H, W, C) -> drop alpha, average RGB to gray
        im = im[..., :3].mean(axis=-1)
    return im.astype(np.float32)

raw_arrays = [load_gray(p) for p in image_paths]         # list of 2D arrays (H, W)
label_arrays = [imageio.imread(p).astype(np.int32) for p in label_paths]
print("example image shape:", raw_arrays[0].shape, "| example label shape:", label_arrays[0].shape)

# Pass lists of 2D arrays (raw_key/label_key=None) so each is treated as a separate 2D
# image (image-collection dataset), not slices of a 3D volume. with_channels=False = grayscale.
ds = default_sam_dataset(
    raw_paths=raw_arrays, raw_key=None,
    label_paths=label_arrays, label_key=None,
    patch_shape=patch_shape,
    with_segmentation_decoder=True,
    with_channels=False,
)
n_val = max(1, int(0.1 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])

# num_workers=0 on Windows notebooks avoids multiprocessing/spawn issues.
train_loader = torch_em.get_data_loader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = torch_em.get_data_loader(val_ds, batch_size=1, shuffle=True, num_workers=0)
print(f"train patches/epoch: {len(train_ds)}  |  val: {len(val_ds)}")


images: 5  |  labels: 5
example image shape: (2160, 2160) | example label shape: (2160, 2160)
train patches/epoch: 90  |  val: 10


In [11]:
# Run fine-tuning. The "gtx3080" configuration matches your 10 GB RTX 3080
# (vit_b + memory-efficient settings); pass configuration=None to auto-detect.
# model_type="vit_b_lm" -> start from the Light Microscopy model, not vanilla SAM.
train_sam_for_configuration(
    name=name,
    train_loader=train_loader,
    val_loader=val_loader,
    configuration="gtx3080",
    model_type="vit_b_lm",
    with_segmentation_decoder=True,
    save_root=str(save_root),
    n_epochs=100,          # early stopping halts sooner once val stops improving
)
print("best checkpoint:", save_root / "checkpoints" / name / "best.pt")


Verifying labels in 'val' dataloader:  20%|██        | 10/50 [00:00<00:00, 52.65it/s]


Start fitting for 9000 iterations /  100 epochs
with 90 iterations per epoch
Training with mixed precision


Epoch 22: average [s/it]: 1.800031, current metric: 0.255336, best metric: 0.186320:  23%|██▎       | 2070/9000 [1:04:20<3:35:24,  1.87s/it]

Stopping training because there has been no improvement for 10 epochs
Finished training after 22 epochs / 2070 iterations.
The best epoch is number 11.
Training took 3863.231900215149 seconds (= 01:64:23 hours)
best checkpoint: c:\Users\isobe\Dropbox (Personal)\Python\image_quantification\finetune\checkpoints\nxd8_vit_b_lm\best.pt


In [12]:
# Step 3 — export the fine-tuned model so it works in the napari GUI / CLI / library,
# then re-open the annotator using it. In the GUI, put this path in
# Embedding Settings -> "custom weights path" (keep Model = Light Microscopy, size = base).
from micro_sam.util import export_custom_sam_model

best_ckpt = save_root / "checkpoints" / name / "best.pt"
finetuned_path = save_root / f"{name}_finetuned.pth"

export_custom_sam_model(
    checkpoint_path=str(best_ckpt),
    model_type="vit_b",              # architecture size only (base); the LM weights are inside best.pt
    save_path=str(finetuned_path),
    with_segmentation_decoder=True,  # keep the decoder so AIS/APG work
)
print("finetuned model exported to:", finetuned_path)


finetuned model exported to: c:\Users\isobe\Dropbox (Personal)\Python\image_quantification\finetune\nxd8_vit_b_lm_finetuned.pth
